# Pre-Training Table

This notebook explores `pre_training_table.parquet`, built by `python main.py pre-training`.

The pre-training table assigns a `p.event_label` to every tracking frame using a ±1 second event window. Each frame is labeled with the type of the nearest event anchor within 1 second, or `"no event"` if no anchor is within range.

**Output column:**

| Column | Description |
|---|---|
| `p.event_label` | Event type from the nearest anchor if within ±1 s, otherwise `"no event"` |

All original tracking columns (`t.*`) are preserved.

## 1. Setup

In [128]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

from driblab.config import MODEL_BASE_DATA_DIR

PRE_TRAINING_PATH = MODEL_BASE_DATA_DIR / "pre_training_table.parquet"

assert PRE_TRAINING_PATH.exists(), (
    f"Missing {PRE_TRAINING_PATH}. Run: python main.py pre-training"
)
print(f"Pre-training table: {PRE_TRAINING_PATH}")

Pre-training table: /Users/nataliaurrea/Documents/IE/MBDS/Term III/Driblab/data/processed/model_base/pre_training_table.parquet


## 2. Load pre-training table

In [129]:
pre_training = pd.read_parquet(PRE_TRAINING_PATH)

t_cols = [c for c in pre_training.columns if c.startswith("t.")]
p_cols = [c for c in pre_training.columns if c.startswith("p.")]

print(f"Rows    : {len(pre_training):,}")
print(f"Columns : {pre_training.shape[1]}  (t.*: {len(t_cols)}, p.*: {len(p_cols)})")
print()
print("Label distribution:")
display(
    pre_training["p.event_label"]
    .value_counts()
    .rename("frames")
    .to_frame()
    .assign(pct=lambda x: (x["frames"] / len(pre_training) * 100).round(2))
)
print()
print("Sample rows:")
display(
    pre_training[
        ["t.match_id", "t.period", "t.frame", "t.Videotimestamp",
         "p.event_label", "t.ball_x", "t.ball_y"]
    ].head(8)
)

Rows    : 1,986,630
Columns : 122  (t.*: 121, p.*: 1)

Label distribution:


,frames,pct
p.event_label,,
no event,1240461,62.44
PASS,576467,29.02
BALL TOUCH,74355,3.74
TACKLE,25445,1.28
BALL RECOVERY,19013,0.96
AERIAL,18497,0.93
TAKEON,18046,0.91
FOUL,14346,0.72



Sample rows:


,t.match_id,t.period,t.frame,t.Videotimestamp,p.event_label,t.ball_x,t.ball_y
0,678949,1,0,1.0,PASS,NaN,NaN
1,678949,1,1,1.1,PASS,NaN,NaN
2,678949,1,2,1.2,PASS,NaN,NaN
3,678949,1,3,1.3,PASS,NaN,NaN
4,678949,1,4,1.4,PASS,NaN,NaN
5,678949,1,5,1.5,PASS,NaN,NaN
6,678949,1,6,1.6,PASS,NaN,NaN
7,678949,1,7,1.7,PASS,NaN,NaN


## 3. Verify `t.Videotimestamp`

`t.Videotimestamp` is already stored by the tracking provider as a sub-second float at 10 Hz: `1.0, 1.1, 1.2, … 1.9, 2.0, 2.1, …` — one unique value per frame. No new column is needed; `t.Videotimestamp` is used directly throughout this notebook as the timestamp for all ±1 s distance calculations.

`t.match_clock` only has whole-second resolution (10 frames share the same `[min, sec]` value) and is not used here.

In [130]:
pre_training["t.Videotimestamp"] = pd.to_numeric(
    pre_training["t.Videotimestamp"], errors="coerce"
)

print("t.Videotimestamp — one unique sub-second value per frame at 10 Hz:")
display(
    pre_training[
        ["t.match_id", "t.period", "t.frame", "t.match_clock", "t.Videotimestamp"]
    ].head(12)
)

t.Videotimestamp — one unique sub-second value per frame at 10 Hz:


,t.match_id,t.period,t.frame,t.match_clock,t.Videotimestamp
0,678949,1,0,"[0,1]",1.0
1,678949,1,1,"[0,1]",1.1
2,678949,1,2,"[0,1]",1.2
3,678949,1,3,"[0,1]",1.3
4,678949,1,4,"[0,1]",1.4
5,678949,1,5,"[0,1]",1.5
6,678949,1,6,"[0,1]",1.6
7,678949,1,7,"[0,1]",1.7
8,678949,1,8,"[0,1]",1.8
9,678949,1,9,"[0,1]",1.9


## 4. Event window labeling

Each tracking frame is labeled with the nearest real event anchor within ±1 second. Frames further than 1 s from every anchor remain `"no event"`. When two event windows overlap, the frame goes to the nearer anchor.

The example below shows what this looks like in the data — frames surrounding a labeled event, with `"no event"` frames marking where the window boundary falls.

#### 4.1 Label column

**`p.event_label`**  
The event type assigned to this tracking frame. Every frame within 1 second of a real event anchor carries that event's type. Frames whose nearest anchor is more than 1 second away are labeled `"no event"`.

> The intermediate columns `p.actual_event_frame` and `p.dist_to_actual_event` are computed during the build step but are not saved to the parquet — only `p.event_label` is written.

In [131]:
# Show a neighborhood of frames around a PASS event with the labeling columns.
# p.dist_to_actual_event is recomputed here from the master join for demonstration —
# it is an intermediate column and is not saved to the output parquet.

MASTER_JOIN_PATH = MODEL_BASE_DATA_DIR / "master_join_table.parquet"

sample    = pre_training[pre_training["p.event_label"] == "PASS"].iloc[50]
match_id  = sample["t.match_id"]
period    = sample["t.period"]
ts_center = float(sample["t.Videotimestamp"])

neighborhood = (
    pre_training[
        (pre_training["t.match_id"] == match_id)
        & (pre_training["t.period"] == period)
        & (pre_training["t.Videotimestamp"] >= ts_center - 2.0)
        & (pre_training["t.Videotimestamp"] <= ts_center + 2.0)
    ]
    .sort_values("t.Videotimestamp")
    .reset_index(drop=True)
)

if MASTER_JOIN_PATH.exists():
    # Load only the 4 columns needed to reconstruct anchor timestamps
    mj = pd.read_parquet(
        MASTER_JOIN_PATH,
        columns=["t.match_id", "t.period", "t.Videotimestamp", "e.event.event_type_name"],
    )
    anchors = (
        mj[
            (mj["t.match_id"] == match_id)
            & (mj["t.period"] == period)
            & (mj["e.event.event_type_name"] != "no event")
        ]
        [["t.Videotimestamp"]]
        .rename(columns={"t.Videotimestamp": "p.actual_event_frame"})
        .sort_values("p.actual_event_frame")
    )

    demo = pd.merge_asof(
        neighborhood,
        anchors,
        left_on="t.Videotimestamp",
        right_on="p.actual_event_frame",
        direction="nearest",
    )
    dist = (demo["t.Videotimestamp"] - demo["p.actual_event_frame"]).abs()
    within = dist <= 1.0
    demo["p.dist_to_actual_event"] = np.where(within, dist.round(3), np.nan)
    demo.loc[~within, "p.actual_event_frame"] = np.nan

    display_cols = [
        "t.frame", "t.Videotimestamp",
        "p.event_label", "p.actual_event_frame", "p.dist_to_actual_event",
        "t.ball_x", "t.ball_y",
    ]
else:
    demo = neighborhood
    display_cols = ["t.frame", "t.Videotimestamp", "p.event_label", "t.ball_x", "t.ball_y"]
    print("(master_join_table.parquet not found — p.dist_to_actual_event not shown)")

print(
    f"Match {match_id}, period {period} — ±2 s around a PASS frame at t={ts_center:.2f} s\n"
    "p.dist_to_actual_event = distance to the nearest event anchor.\n"
    "'no event' frames have NaN — their nearest anchor is more than 1 s away."
)
display(demo[display_cols])

Match 678949, period 1 — ±2 s around a PASS frame at t=12.42 s
p.dist_to_actual_event = distance to the nearest event anchor.
'no event' frames have NaN — their nearest anchor is more than 1 s away.


,t.frame,t.Videotimestamp,p.event_label,p.actual_event_frame,p.dist_to_actual_event,t.ball_x,t.ball_y
0,95,10.42,AERIAL,9.92,0.5,NaN,NaN
1,96,10.52,AERIAL,9.92,0.6,NaN,NaN
2,97,10.62,AERIAL,9.92,0.7,NaN,NaN
3,98,10.72,PASS,11.52,0.8,64.55,56.39
4,99,10.82,PASS,11.52,0.7,63.62,57.63
5,100,10.92,PASS,11.52,0.6,62.70,58.87
6,101,11.02,PASS,11.52,0.5,67.68,59.01
7,102,11.12,PASS,11.52,0.4,NaN,NaN
8,103,11.22,PASS,11.52,0.3,NaN,NaN
9,104,11.32,PASS,11.52,0.2,NaN,NaN


## 5. Label statistics

In [132]:
print("Label distribution (pre-training table):")
display(
    pre_training["p.event_label"]
    .value_counts()
    .rename("frames")
    .to_frame()
    .assign(pct=lambda x: (x["frames"] / len(pre_training) * 100).round(2))
)

n_event    = int((pre_training["p.event_label"] != "no event").sum())
n_no_event = int((pre_training["p.event_label"] == "no event").sum())
print()
print(f"Event-labeled frames : {n_event:,}  ({n_event / len(pre_training) * 100:.1f}%)")
print(f"No-event frames      : {n_no_event:,}  ({n_no_event / len(pre_training) * 100:.1f}%)")

Label distribution (pre-training table):


,frames,pct
p.event_label,,
no event,1240461,62.44
PASS,576467,29.02
BALL TOUCH,74355,3.74
TACKLE,25445,1.28
BALL RECOVERY,19013,0.96
AERIAL,18497,0.93
TAKEON,18046,0.91
FOUL,14346,0.72



Event-labeled frames : 746,169  (37.6%)
No-event frames      : 1,240,461  (62.4%)


## 6. Overlap logic — what happens when two event windows collide

Two events whose anchor frames are less than `2 × WINDOW_SEC = 2.0 s` apart will have overlapping ±1 s windows. Any tracking frame inside that overlap zone is within 1 second of **both** events.

**How the overlap is resolved:** `pd.merge_asof(direction="nearest")` assigns each tracking frame to the single event whose anchor timestamp is closest to the frame's `p.tracking_sec`. The further event is never considered for that frame.

**Example:**  
- Event A (PASS) anchored at `t = 45.2 s`  
- Event B (BALL TOUCH) anchored at `t = 46.0 s` (gap = 0.8 s → windows overlap by 1.2 s)  
- Overlap zone: frames between `t = 45.0 s` and `t = 46.0 s` (the intersection of [44.2, 46.2] and [45.0, 47.0])  

For a frame at `t = 45.5 s`:  
- Distance to A = |45.5 − 45.2| = 0.3 s  
- Distance to B = |45.5 − 46.0| = 0.5 s  
- **Result: labeled PASS** (A is closer, and 0.3 s ≤ 1.0 s)  

For a frame at `t = 45.7 s`:  
- Distance to A = 0.5 s, distance to B = 0.3 s  
- **Result: labeled BALL TOUCH** (B is closer)  

The **midpoint** between two event anchors is the exact split boundary. Frames before the midpoint go to the earlier event; frames at or after the midpoint go to the later event. In the case of a perfect tie (frame exactly equidistant) `merge_asof` picks the event that appears first in the sorted anchor list (i.e., the earlier-anchored one).

The preview below shows 1.5 s either side of the first PASS event — if the window is open at the edge it means no other event is within 1 s on that side.

In [133]:
# Find the first transition between two different real event labels
# to show how overlapping ±1 s windows are resolved (nearest anchor wins)

df_sorted = (
    pre_training
    .sort_values(["t.match_id", "t.period", "t.Videotimestamp"])
    .reset_index(drop=True)
)

prev_label = df_sorted["p.event_label"].shift()
curr_label = df_sorted["p.event_label"]

transition_mask = (
    (curr_label != prev_label)
    & (curr_label != "no event")
    & (prev_label != "no event")
    & (df_sorted["t.match_id"] == df_sorted["t.match_id"].shift())
    & (df_sorted["t.period"] == df_sorted["t.period"].shift())
)

if transition_mask.any():
    boundary_idx  = int(transition_mask.idxmax())
    boundary_row  = df_sorted.iloc[boundary_idx]
    match_id      = boundary_row["t.match_id"]
    period        = boundary_row["t.period"]
    ts_boundary   = float(boundary_row["t.Videotimestamp"])
    label_before  = str(prev_label.iloc[boundary_idx])
    label_after   = str(curr_label.iloc[boundary_idx])

    window = (
        df_sorted[
            (df_sorted["t.match_id"] == match_id)
            & (df_sorted["t.period"] == period)
            & (df_sorted["t.Videotimestamp"] >= ts_boundary - 1.5)
            & (df_sorted["t.Videotimestamp"] <= ts_boundary + 1.5)
        ]
        [["t.frame", "t.Videotimestamp", "p.event_label", "t.ball_x", "t.ball_y"]]
    )

    print(f"Transition: {label_before!r} → {label_after!r} at t={ts_boundary:.2f} s")
    print(f"(match {match_id}, period {period})")
    print("The boundary frame is the midpoint between the two event anchors.")
    display(window)
else:
    print("No direct event-to-event transition found in this dataset.")

Transition: 'AERIAL' → 'PASS' at t=10.72 s
(match 678949, period 1)
The boundary frame is the midpoint between the two event anchors.


,t.frame,t.Videotimestamp,p.event_label,t.ball_x,t.ball_y
83,83,9.22,AERIAL,15.39,106.33
84,84,9.32,AERIAL,NaN,NaN
85,85,9.42,AERIAL,NaN,NaN
86,86,9.52,AERIAL,NaN,NaN
87,87,9.62,AERIAL,NaN,NaN
88,88,9.72,AERIAL,NaN,NaN
89,89,9.82,AERIAL,NaN,NaN
90,90,9.92,AERIAL,NaN,NaN
91,91,10.02,AERIAL,NaN,NaN
92,92,10.12,AERIAL,66.22,22.61


## 7. Output

In [134]:
print(f"File  : {PRE_TRAINING_PATH}")
print(f"Rows  : {len(pre_training):,}")
print(f"Cols  : {pre_training.shape[1]}  "
      f"(t.*: {sum(c.startswith('t.') for c in pre_training.columns)}, p.*: 1)")
print()
print("Built by: python main.py pre-training")

File  : /Users/nataliaurrea/Documents/IE/MBDS/Term III/Driblab/data/processed/model_base/pre_training_table.parquet
Rows  : 1,986,630
Cols  : 122  (t.*: 121, p.*: 1)

Built by: python main.py pre-training


In [137]:
print(f"Output parquet: {len(pre_training):,} rows × {pre_training.shape[1]} columns")
print(f"  {sum(c.startswith('t.') for c in pre_training.columns)} tracking columns (t.*)")
print(f"  1 label column: p.event_label")
print()
print("Sample rows (key columns — as stored in the parquet):")
display(pre_training.head(12))

Output parquet: 1,986,630 rows × 122 columns
  121 tracking columns (t.*)
  1 label column: p.event_label

Sample rows (key columns — as stored in the parquet):


,t.match_id,t.match_clock,t.frame,t.Videotimestamp,t.period,t.cam,t.ball_x,t.ball_y,t.ball_z,t.player_count,t.visible_player_count,t.player_01_team_id,t.player_01_id,t.player_01_x,t.player_01_y,t.player_01_visible,t.player_02_team_id,t.player_02_id,t.player_02_x,t.player_02_y,t.player_02_visible,t.player_03_team_id,t.player_03_id,t.player_03_x,t.player_03_y,t.player_03_visible,t.player_04_team_id,t.player_04_id,t.player_04_x,t.player_04_y,t.player_04_visible,t.player_05_team_id,t.player_05_id,t.player_05_x,t.player_05_y,t.player_05_visible,t.player_06_team_id,t.player_06_id,t.player_06_x,t.player_06_y,t.player_06_visible,t.player_07_team_id,t.player_07_id,t.player_07_x,t.player_07_y,t.player_07_visible,t.player_08_team_id,t.player_08_id,t.player_08_x,t.player_08_y,t.player_08_visible,t.player_09_team_id,t.player_09_id,t.player_09_x,t.player_09_y,t.player_09_visible,t.player_10_team_id,t.player_10_id,t.player_10_x,t.player_10_y,t.player_10_visible,t.player_11_team_id,t.player_11_id,t.player_11_x,t.player_11_y,t.player_11_visible,t.player_12_team_id,t.player_12_id,t.player_12_x,t.player_12_y,t.player_12_visible,t.player_13_team_id,t.player_13_id,t.player_13_x,t.player_13_y,t.player_13_visible,t.player_14_team_id,t.player_14_id,t.player_14_x,t.player_14_y,t.player_14_visible,t.player_15_team_id,t.player_15_id,t.player_15_x,t.player_15_y,t.player_15_visible,t.player_16_team_id,t.player_16_id,t.player_16_x,t.player_16_y,t.player_16_visible,t.player_17_team_id,t.player_17_id,t.player_17_x,t.player_17_y,t.player_17_visible,t.player_18_team_id,t.player_18_id,t.player_18_x,t.player_18_y,t.player_18_visible,t.player_19_team_id,t.player_19_id,t.player_19_x,t.player_19_y,t.player_19_visible,t.player_20_team_id,t.player_20_id,t.player_20_x,t.player_20_y,t.player_20_visible,t.player_21_team_id,t.player_21_id,t.player_21_x,t.player_21_y,t.player_21_visible,t.player_22_team_id,t.player_22_id,t.player_22_x,t.player_22_y,t.player_22_visible,p.event_label
0,678949,"[0,1]",0,1.0,1,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,PASS
1,678949,"[0,1]",1,1.1,1,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,PASS
2,678949,"[0,1]",2,1.2,1,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,PASS
3,678949,"[0,1]",3,1.3,1,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN

## 8. Frame timing audit — are `t.Videotimestamp` gaps uniform?

`t.Videotimestamp` is the raw value from the tracking provider and is stored unchanged by the ETL. It is **not** reconstructed — unlike the internal `_tracking_match_clock_seconds` which is built as `whole_second + cumcount / fps`.

Gaps that deviate from 0.10 s in `p.event_dist_sec` reflect genuine non-uniformity in the provider's timestamps. Two likely causes:

1. **Dropped frame** — the provider skipped one frame, leaving a 0.20 s gap.
2. **25fps source video** — tracking derived from 25fps video sampled every ~2.5 frames alternates between 0.08 s and 0.12 s gaps to average 10 Hz.

This does **not** break the ±1 s labeling: both sides of the distance subtraction (`t.Videotimestamp` of each frame and `p.event_frame_sec` of the anchor) come from the same raw values, so distances are accurate.

In [136]:
# Compute frame-to-frame t.Videotimestamp gaps per (match, period)
# Expected: all gaps = 0.1 s. Any other value = irregular frame timing in raw data.
audit_rows = []
for (mid, per), grp in pre_training.groupby(["t.match_id", "t.period"]):
    ts = grp.sort_values("t.Videotimestamp")["t.Videotimestamp"].values
    gaps = np.diff(ts).round(4)  # round to 4dp to absorb float noise
    unique_gaps, counts = np.unique(gaps, return_counts=True)
    n_irregular = int((~np.isclose(gaps, 0.1, atol=1e-3)).sum())
    irregular_vals = sorted(set(gaps[~np.isclose(gaps, 0.1, atol=1e-3)].round(3)))
    audit_rows.append({
        "match_id": mid,
        "period": per,
        "total_frames": len(ts),
        "n_gaps": len(gaps),
        "n_irregular": n_irregular,
        "pct_irregular": round(n_irregular / len(gaps) * 100, 2) if len(gaps) else 0,
        "irregular_gap_values": irregular_vals,
    })

audit_df = pd.DataFrame(audit_rows)
print(f"Periods with ALL uniform 0.1 s gaps : {(audit_df['n_irregular'] == 0).sum()} / {len(audit_df)}")
print(f"Periods with ANY irregular gap      : {(audit_df['n_irregular'] > 0).sum()} / {len(audit_df)}")
print(f"Total irregular gaps across dataset : {audit_df['n_irregular'].sum():,}")
print()
print("Breakdown per (match, period):")
display(
    audit_df[["match_id", "period", "total_frames", "n_irregular", "pct_irregular", "irregular_gap_values"]]
    .sort_values("n_irregular", ascending=False)
)

Periods with ALL uniform 0.1 s gaps : 0 / 66
Periods with ANY irregular gap      : 66 / 66
Total irregular gaps across dataset : 17,417

Breakdown per (match, period):


,match_id,period,total_frames,n_irregular,pct_irregular,irregular_gap_values
65,745399,2,29044,408,1.40,"[0.02, 0.08]"
42,684147,1,29054,391,1.35,"[0.02, 0.08, 0.12]"
43,684147,2,30084,386,1.28,"[0.02, 0.08, 1.1]"
44,689340,1,28536,377,1.32,"[0.02, 0.08, 0.62, 1.1]"
21,682815,2,32172,374,1.16,"[0.02, 0.08, 1.1, 1.6, 5.1]"
...,...,...,...,...,...,...
10,679088,1,31262,179,0.57,"[0.02, 0.04, 0.06, 0.08, 0.12, 0.6]"
28,683261,1,30784,179,0.58,"[0.02, 0.04, 0.06, 0.08, 0.12, 0.56]"
9,679075,2,30134,178,0.59,"[0.02, 0.04, 0.06, 0.08, 0.12]"
56,713910,1,29072,175,0.60,"[0.02, 0.04, 0.06, 0.08, 0.12, 0.54]"
